# S3 Vectors: Metadata Filtering & Benchmark

This notebook stores video embeddings in **Amazon S3 Vectors** and performs search.

The key strength of S3 Vectors is **simplicity and cost efficiency**. Just create a Vector Bucket and Index to get started — no separate security policies or VPC configuration needed, as access control is handled through existing IAM policies.

Topics covered:
1. Vector ingest and speed measurement (key-based upsert)
2. Metadata filtering search
3. Search latency benchmark (p50/p95/p99)
4. Results saved to `benchmark_s3vectors.json` for comparison in notebook 04

**Prerequisites:** Complete `01_setup_and_embeddings.ipynb`

## 0. Setup

In [ ]:
%pip install -r requirements.txt -Uq

In [ ]:
import json, time
import boto3

with open("config.json") as f:
    config = json.load(f)

REGION = config["REGION"]
S3_BUCKET = config["S3_BUCKET"]
EMBEDDINGS_S3_PREFIX = config["EMBEDDINGS_S3_PREFIX"]
EMBEDDING_DIMENSION = config["EMBEDDING_DIMENSION"]
VIDEO_IDS = config["VIDEO_IDS"]
S3_VECTOR_BUCKET_FROM_CONFIG = config.get("S3_VECTOR_BUCKET_NAME", "")

session = boto3.Session()
s3 = session.client("s3", region_name=REGION)
s3v = session.client("s3vectors", region_name=REGION)

print(f"Vector dimension: {EMBEDDING_DIMENSION}, Videos: {len(VIDEO_IDS)}")

### Parameters

| Parameter | Description | Recommended |
|-----------|-------------|-------------|
| `VECTOR_BUCKET_NAME` | S3 Vector Bucket name (equivalent to a DB) | Use workshop-provided value |
| `INDEX_NAME` | Vector index name (equivalent to a table) | Any name |
| `INGEST_BATCH_SIZE` | Vectors per `put_vectors` call. API max is 500 | 50-500 |
| `SEARCH_K` | Number of nearest neighbors to return | 5, 10, 20 |

> **S3 Vectors structure**: Vector Bucket -> Index -> Vectors (similar to DB -> Table -> Rows)
>
> Each vector consists of a **key** (unique ID), **data** (embedding), and **metadata** (filterable attributes). Re-storing with the same key automatically upserts.

In [ ]:
VECTOR_BUCKET_NAME = S3_VECTOR_BUCKET_FROM_CONFIG or "video-vector-store"
INDEX_NAME = "video-embeddings"
INGEST_BATCH_SIZE = 500
SEARCH_K = 5

## 1. Connect to S3 Vectors

Check or create the Vector Bucket and Index. Unlike OpenSearch, no separate security policy configuration is required.

> **Difference from OpenSearch**: OpenSearch requires 3 separate policies (encryption, network, data access), while S3 Vectors uses existing IAM policies for access control.

In [ ]:
# Check/create Vector Bucket
bucket_exists = any(b["vectorBucketName"] == VECTOR_BUCKET_NAME
                    for b in s3v.list_vector_buckets().get("vectorBuckets", []))
if not bucket_exists:
    s3v.create_vector_bucket(vectorBucketName=VECTOR_BUCKET_NAME)
print(f"Vector Bucket: {VECTOR_BUCKET_NAME} ({'existing' if bucket_exists else 'newly created'})")

# Check/create Index
try:
    s3v.get_index(vectorBucketName=VECTOR_BUCKET_NAME, indexName=INDEX_NAME)
    print(f"Index: {INDEX_NAME} (existing)")
except:
    s3v.create_index(vectorBucketName=VECTOR_BUCKET_NAME, indexName=INDEX_NAME,
                     dimension=EMBEDDING_DIMENSION, distanceMetric="cosine", dataType="float32")
    print(f"Index: {INDEX_NAME} (newly created, {EMBEDDING_DIMENSION}-dim, cosine)")

## 2. Ingest Embeddings

Load embeddings from S3 and ingest into S3 Vectors.

> **Batch size impact**: In blog tests, increasing batch size from 50 to 500 improved ingest time ~3x (13.46s -> 4.12s). Using the API max of 500 is recommended.
>
> **Key-based upsert**: S3 Vectors automatically updates when storing with the same key. No separate delete step needed when regenerating embeddings.

In [ ]:
def load_embeddings_from_s3():
    """Load embeddings from S3 (searches multiple paths, handles Workshop 1 and 4 formats)."""
    vectors, found_files = [], []
    for prefix in [EMBEDDINGS_S3_PREFIX, 'embeddings/videos/', 'embeddings/']:
        resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=prefix, MaxKeys=200)
        for obj in resp.get('Contents', []):
            if obj['Key'].endswith('.json') and '/config.json' not in obj['Key']:
                found_files.append(obj['Key'])
        if found_files: break

    # Skip output.json/manifest.json if consolidated files exist (avoid duplicates)
    consolidated = [f for f in found_files if f.split('/')[-1] not in ('output.json', 'manifest.json')]
    if consolidated:
        found_files = consolidated

    for s3_key in found_files:
        try: data = json.loads(s3.get_object(Bucket=S3_BUCKET, Key=s3_key)['Body'].read().decode())
        except: continue
        segs = data.get('vectors', data.get('data', []))
        vid = data.get('video_id', '')
        if not vid:
            basename = s3_key.split('/')[-1].replace('.json', '')
            if basename == 'output':
                import os as _os
                s3_uri = data.get('inputVideo', {}).get('s3Uri', '')
                if not s3_uri:
                    s3_uri = data.get('video', {}).get('mediaSource', {}).get('s3Location', {}).get('uri', '')
                if s3_uri:
                    vid = _os.path.splitext(_os.path.basename(s3_uri))[0]
                else:
                    vid = s3_key.split('/')[-3]
            else:
                vid = basename
        for seg in segs:
            if 'embedding' not in seg: continue
            vectors.append({
                'video_id': vid, 'embedding': seg['embedding'],
                'start_sec': seg.get('startSec', seg.get('start_time', 0)),
                'end_sec': seg.get('endSec', seg.get('end_time', 0)),
                'scope': seg.get('embeddingScope', seg.get('embedding_scope', 'clip'))
            })
    return vectors

vectors = load_embeddings_from_s3()
print(f"Loaded {len(vectors)} vectors from S3")

In [ ]:
# Ingest with timing (upsert — safe to re-run)
print(f"   Ingesting: {len(vectors)} vectors, batch_size={INGEST_BATCH_SIZE}...")
ingest_start = time.time()
total = 0
for b in range(0, len(vectors), INGEST_BATCH_SIZE):
    batch = vectors[b:b + INGEST_BATCH_SIZE]
    items = [{
        "key": f"{v['video_id']}_{v['scope']}_{v['start_sec']}_{b+i}",
        "data": {"float32": v["embedding"]},
        "metadata": {"videoId": v["video_id"], "startSec": str(v["start_sec"]),
                     "endSec": str(v["end_sec"]), "scope": v["scope"]}
    } for i, v in enumerate(batch)]
    s3v.put_vectors(vectorBucketName=VECTOR_BUCKET_NAME, indexName=INDEX_NAME, vectors=items)
    total += len(batch)
ingest_time = time.time() - ingest_start

print(f"Ingest complete: {total} vectors, {ingest_time:.2f}s")

## 3. Metadata Filtering

Hands-on with S3 Vectors **metadata filtering**. Use metadata attached during vector storage to narrow the search scope.

| Filter | Use Case |
|--------|----------|
| `videoId = "abc"` | Search within a specific video only |
| `scope = "clip"` | Search clip-level segments only (exclude asset-level) |

> **Pre-filter vs Post-filter**: S3 Vectors metadata filtering is **pre-filter**. It narrows candidates before similarity computation, so it always returns k results.
>
> **Difference from OpenSearch**: OpenSearch supports **full-text search** (BM25) in addition to metadata filtering. S3 Vectors only supports exact value matching on metadata.

In [ ]:
# Text-to-embedding
from botocore.config import Config
bedrock = session.client("bedrock-runtime", region_name=REGION, config=Config(signature_version="v4"))

def embed_text(query):
    resp = bedrock.invoke_model(
        modelId="us.twelvelabs.marengo-embed-3-0-v1:0",
        body=json.dumps({"inputType": "text", "text": {"inputText": query}})
    )
    return json.loads(resp["body"].read())["data"][0]["embedding"]

print("Text embedding function ready")

In [ ]:
# ==============================
# Enter your search query
# ==============================
SEARCH_QUERY = "goal scene in a soccer match"  # ← Try different queries!

query_vector = embed_text(SEARCH_QUERY)
print(f"Query: '{SEARCH_QUERY}' -> {len(query_vector)}-dim vector")

In [ ]:
def search_and_print(label, filter_expr=None):
    kwargs = {"vectorBucketName": VECTOR_BUCKET_NAME, "indexName": INDEX_NAME,
              "queryVector": {"float32": query_vector}, "topK": SEARCH_K,
              "returnMetadata": True, "returnDistance": True}
    if filter_expr: kwargs["filter"] = filter_expr
    t0 = time.time()
    result = s3v.query_vectors(**kwargs)
    latency = (time.time() - t0) * 1000
    print(f"\n{label} (latency={latency:.1f}ms):")
    for i, vec in enumerate(result.get("vectors", []), 1):
        m = vec.get("metadata", {})
        print(f"  {i}. dist={vec.get('distance','N/A'):<8} | {m.get('videoId','?')} | {m.get('startSec','?')}s-{m.get('endSec','?')}s | {m.get('scope','?')}")

# 1) No filter (default)
search_and_print("No filter")

# 2) Clip-level only
search_and_print("Clip-level only", {"scope": {"$eq": "clip"}})

# 3) Specific video only
first_video = VIDEO_IDS[0]
search_and_print(f"Video '{first_video}' only", {"videoId": {"$eq": first_video}})

In [ ]:
# Video playback
from IPython.display import display, HTML

def build_video_mapping():
    mapping = {}
    for prefix in ['videos/', 'vectordb-workshop/videos/']:
        resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=prefix, MaxKeys=200)
        for obj in resp.get('Contents', []):
            if obj['Key'].endswith('.mp4'):
                import os; mapping[os.path.splitext(os.path.basename(obj['Key']))[0]] = obj['Key']
    for prefix in ['embeddings/videos/', 'vectordb-workshop/embeddings/']:
        resp = s3.list_objects_v2(Bucket=S3_BUCKET, Prefix=prefix, MaxKeys=500)
        for obj in resp.get('Contents', []):
            if obj['Key'].endswith('output.json'):
                try:
                    data = json.loads(s3.get_object(Bucket=S3_BUCKET, Key=obj['Key'])['Body'].read().decode())
                    s3_uri = data.get('inputVideo', {}).get('s3Uri', '')
                    if s3_uri:
                        parts = obj['Key'].split('/')
                        mapping[parts[2] if len(parts) > 2 else parts[-2]] = s3_uri.replace(f's3://{S3_BUCKET}/', '')
                except: continue
    return mapping

video_mapping = build_video_mapping()

def play_result(metadata):
    vid = metadata.get('videoId', metadata.get('video_id', ''))
    start = metadata.get('startSec', metadata.get('start_sec', '0'))
    key = video_mapping.get(vid)
    if key:
        url = s3.generate_presigned_url('get_object', Params={'Bucket': S3_BUCKET, 'Key': key}, ExpiresIn=3600)
        display(HTML(f'<p>{vid} ({start}s~)</p><video width="640" controls><source src="{url}#t={start}" type="video/mp4"></video>'))
    else:
        print(f"Video not found: '{vid}'")

# Play top 1 from search results
top_result = s3v.query_vectors(
    vectorBucketName=VECTOR_BUCKET_NAME, indexName=INDEX_NAME,
    queryVector={"float32": query_vector}, topK=SEARCH_K,
    returnMetadata=True, returnDistance=True
)
vecs = top_result.get("vectors", [])
if vecs:
    play_result(vecs[0].get("metadata", {}))
else:
    print(f"No search results")

## 4. Search Benchmark

Measure search latency. Results are saved to `benchmark_s3vectors.json` for comparison with OpenSearch in notebook 04.

> **Reference numbers from blog**: S3 Vectors measured p50~65ms, p95~181ms at k=5. Slower than OpenSearch (p50~25ms), but the simplicity and cost efficiency are its strengths.

In [ ]:
BENCHMARK_ITERATIONS = 20
BENCHMARK_QUERY_COUNT = 5

In [ ]:
import numpy as np

query_vectors_bench = [v["embedding"] for v in vectors if v["scope"] == "asset"][:BENCHMARK_QUERY_COUNT]
if not query_vectors_bench: query_vectors_bench = [vectors[0]["embedding"]]

print(f"Benchmark: {len(query_vectors_bench)} queries x {BENCHMARK_ITERATIONS} iterations\n")

benchmark_results = {"service": "S3 Vectors", "vector_count": len(vectors),
                     "ingest_time": ingest_time, "ingest_batch_size": INGEST_BATCH_SIZE, "search": {}}

for k_val in [5, 10, 20]:
    latencies = []
    for _ in range(BENCHMARK_ITERATIONS):
        for qv in query_vectors_bench:
            t0 = time.time()
            s3v.query_vectors(vectorBucketName=VECTOR_BUCKET_NAME, indexName=INDEX_NAME,
                              queryVector={"float32": qv}, topK=k_val)
            latencies.append((time.time() - t0) * 1000)
    latencies.sort()
    n = len(latencies)
    stats = {"p50": latencies[n//2], "p95": latencies[int(n*0.95)], "p99": latencies[int(n*0.99)], "avg": np.mean(latencies)}
    benchmark_results["search"][f"k={k_val}"] = stats
    print(f"  k={k_val:<3} | p50={stats['p50']:.1f}ms | p95={stats['p95']:.1f}ms | p99={stats['p99']:.1f}ms | avg={stats['avg']:.1f}ms")

print("\nBenchmark complete")

In [ ]:
# Save results (used in 04_comparison.ipynb)
with open("benchmark_s3vectors.json", "w") as f:
    json.dump(benchmark_results, f, indent=2)

print(f"benchmark_s3vectors.json saved")
print(f"   Ingest: {ingest_time:.2f}s ({len(vectors)} vectors, batch={INGEST_BATCH_SIZE})")
print(f"   Search p50 (k=5): {benchmark_results['search']['k=5']['p50']:.1f}ms")
print(f"\n   Next: Run 04_comparison.ipynb to compare with OpenSearch")